In [0]:
df = spark.table("olist.gold.order_analysis").limit(20000)

df_fragmentado = df.repartition(20)

In [0]:
from pyspark.sql import functions as F

(
    df_fragmentado
    .select(F.spark_partition_id().alias("partition_id"))
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
    .display()
)

In [0]:
df = spark.table("olist.gold.order_analysis").limit(20000)

df_fragmentado = df.repartition(20)

(
    df_fragmentado.write
    .format("delta")
    .mode("overwrite")
    .option("maxRecordsPerFile", 1000)
    .saveAsTable("olist.gold.optimization_test")
)

In [0]:
%sql
DESCRIBE DETAIL olist.gold.optimization_test;

In [0]:
%sql
optimize olist.gold.optimization_test zorder by (customer_id)

In [0]:
%sql
describe detail olist.gold.optimization_test

In [0]:
%sql
describe history olist.gold.optimization_test

In [0]:
%sql
VACUUM olist.gold.optimization_test DRY RUN;

In [0]:
%sql
SELECT COUNT(*)
FROM olist.gold.optimization_test VERSION AS OF 0;


In [0]:
%sql
SHOW TBLPROPERTIES olist.gold.optimization_test;


# Resumen - Optimization

En este notebook se analizaron y probaron técnicas de optimización y mantenimiento de tablas Delta Lake.

El objetivo principal fue comprender cuándo tiene sentido aplicar `OPTIMIZE`, cómo funciona la compactación de archivos, qué papel cumple `ZORDER` y cómo se relaciona `VACUUM` con Time Travel y la retención de archivos históricos.

---

## 1. Inspección inicial de tablas Delta

Se utilizó:

`DESCRIBE DETAIL`

para analizar el estado físico de las tablas Delta.

En `olist.gold.order_analysis` se observó:

- formato: `delta`;
- tamaño aproximado: **9 MB**;
- número de archivos: **1**;
- sin particiones;
- sin clustering;
- compresión Parquet mediante `zstd`;
- soporte para Deletion Vectors.

### Conclusión

La tabla `order_analysis` ya se encontraba físicamente compactada en un solo archivo, por lo que ejecutar `OPTIMIZE` sobre ella no aportaría una mejora relevante.

Esto permitió comprobar que la optimización debe aplicarse cuando existe una necesidad real y no simplemente ejecutarse sobre todas las tablas.

---

## 2. Creación de una tabla experimental fragmentada

Para demostrar el funcionamiento de `OPTIMIZE`, se creó:

`olist.gold.optimization_test`

La tabla se generó mediante PySpark utilizando una muestra de **20.000 registros** de `order_analysis`.

Se utilizó `repartition(20)` y se limitó el número de registros por archivo para provocar intencionadamente una escritura fragmentada.

### Estado inicial

Mediante `DESCRIBE DETAIL` se comprobó:

- Número de archivos: **20**
- Tamaño total: **2.380.235 bytes**

La tabla presentaba así un escenario de múltiples archivos pequeños, adecuado para probar compactación.

---

## 3. OPTIMIZE y ZORDER

Se ejecutó:

`OPTIMIZE ... ZORDER BY (customer_id)`

El objetivo fue combinar dos conceptos:

- compactación de archivos mediante `OPTIMIZE`;
- organización física de los datos mediante `ZORDER`.

### Resultado

Después de la operación:

- archivos iniciales: **20**
- archivos finales: **1**
- tamaño final: **1.978.636 bytes**
- archivos eliminados de la versión activa: **20**
- archivos nuevos generados: **1**

El Transaction Log registró la operación `OPTIMIZE` y confirmó que se utilizó:

`ZORDER BY (customer_id)`

---

## 4. Interpretación de OPTIMIZE

`OPTIMIZE` permite reducir el problema de los archivos pequeños mediante compactación.

Conceptualmente, el proceso realizado fue:

**20 archivos pequeños → OPTIMIZE → 1 archivo compactado**

Esto puede reducir el overhead asociado a la apertura y gestión de numerosos archivos durante las consultas.

En este experimento el dataset es pequeño, por lo que el objetivo no fue demostrar una mejora significativa en tiempo de ejecución, sino comprender el comportamiento físico de Delta Lake.

---

## 5. ZORDER y Data Skipping

`ZORDER` permite organizar los datos teniendo en cuenta una o varias columnas utilizadas frecuentemente en filtros.

En este caso se utilizó:

`customer_id`

El objetivo de este tipo de organización es facilitar el **data skipping**, permitiendo que Databricks descarte archivos que no contienen los valores necesarios para una consulta.

En la prueba realizada, `OPTIMIZE` redujo la tabla a un único archivo.

Por este motivo, no es posible observar una mejora significativa de data skipping, ya que con un solo archivo Databricks debe consultar ese archivo independientemente del valor de `customer_id`.

El beneficio de `ZORDER` sería más visible en una tabla de mayor tamaño que mantenga múltiples archivos después de la optimización.

---

## 6. VACUUM

Después de `OPTIMIZE`, los 20 archivos anteriores dejaron de formar parte de la versión activa de la tabla.

Sin embargo, estos archivos no se eliminan inmediatamente porque pueden ser necesarios para reconstruir versiones anteriores mediante Time Travel.

Se utilizó:

`VACUUM ... DRY RUN`

para comprobar qué archivos podían eliminarse sin realizar ninguna eliminación física.

### Resultado

`DRY RUN` devolvió:

**0 archivos candidatos**

Esto significa que los archivos antiguos todavía se encuentran protegidos por el período de retención de Delta Lake.

---

## 7. Retención de archivos

Se consultaron las propiedades de la tabla mediante:

`SHOW TBLPROPERTIES`

La propiedad:

`delta.deletedFileRetentionDuration`

no se encontraba definida explícitamente.

Por tanto, se utiliza la configuración de retención predeterminada de Delta Lake.

En este contexto, los archivos eliminados lógicamente todavía no han superado la ventana de retención necesaria para ser eliminados físicamente mediante `VACUUM`.

---

## 8. Prueba de VACUUM con retención reducida

Se intentó realizar una prueba experimental utilizando:

`VACUUM ... RETAIN 0 HOURS`

Databricks bloqueó la operación mediante su mecanismo de protección de retención.

También se intentó desactivar la comprobación de seguridad, pero el entorno Serverless no permite modificar esa configuración.

### Conclusión

El entorno Serverless impide realizar una eliminación agresiva inmediata de archivos antiguos.

Esta protección evita que una operación `VACUUM` incorrectamente configurada pueda eliminar archivos necesarios para operaciones concurrentes o para Time Travel.

---

## 9. Relación entre VACUUM y Time Travel

La prueba permitió reforzar la relación entre los conceptos estudiados en los notebooks 09 y 10.

Time Travel depende de dos elementos:

- el Transaction Log;
- los archivos físicos correspondientes a las versiones históricas.

`VACUUM` puede eliminar archivos físicos que ya no forman parte de la versión actual cuando superan el período de retención.

Por tanto, aunque una versión pueda seguir apareciendo en el historial lógico de Delta Lake, podría dejar de ser consultable si los archivos físicos necesarios fueron eliminados.

Conceptualmente:

**Transaction Log + archivos históricos → Time Travel**

y posteriormente:

**VACUUM → eliminación de archivos históricos elegibles → reducción de disponibilidad de versiones antiguas**

---

# Resultado final

En este notebook se practicaron y analizaron:

- `DESCRIBE DETAIL`
- número y tamaño de archivos Delta;
- fragmentación en archivos pequeños;
- PySpark `repartition`;
- `OPTIMIZE`;
- compactación de archivos;
- `ZORDER`;
- data skipping;
- `DESCRIBE HISTORY`;
- `VACUUM DRY RUN`;
- períodos de retención;
- relación entre `VACUUM` y Time Travel.

## Conclusión

La optimización de tablas Delta debe realizarse en función de las características reales de los datos.

Una tabla pequeña almacenada en un único archivo no necesita ser optimizada únicamente por utilizar Delta Lake.

En cambio, cuando existen numerosos archivos pequeños, `OPTIMIZE` puede compactarlos y reducir el overhead de lectura.

`ZORDER` puede mejorar el acceso a los datos cuando existen patrones frecuentes de filtrado, especialmente en tablas suficientemente grandes como para beneficiarse del data skipping.

Finalmente, `VACUUM` cumple una función distinta: elimina archivos históricos que ya no son necesarios después del período de retención, por lo que debe utilizarse considerando su impacto sobre Time Travel y recuperación histórica.